# CreditWise Analytics — Model Evaluation & Visualization Notebook
Generates **Confusion Matrix Heatmaps** (`YlGnBu`) and **Accuracy, Precision, Recall, F1-score** visualizations for **1. Decision Tree** and **2. Logistic Regression** trained on `public/assets/aistudio/loan_approval_hybrid_dataset_20711486.csv`.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

CSV_PATH = "public/assets/aistudio/loan_approval_hybrid_dataset_20711486.csv"
df = pd.read_csv(CSV_PATH)
print("Dataset shape:", df.shape)

y = df["Loan_Status"].astype(int)
X = df.drop(columns=["Loan_Status"]).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Train split: {len(X_train)} | Test split: {len(X_test)}")

Dataset shape: (10000, 16)
Train split: 8000 | Test split: 2000


In [2]:
NUM_COLS = [
    "Applicant_Income", "Coapplicant_Income", "Credit_Score", "DTI_Ratio",
    "Loan_Amount", "Loan_Tenure", "Existing_Loans", "Previous_Loan_Defaults",
    "Employment_Stability", "Monthly_Expenses", "Savings_Balance",
    "EMI_to_Income_Ratio", "Collateral_Value"
]
CAT_COLS = ["Payment_History", "Loan_Purpose"]

def engineer_features(frame):
    out = frame.copy()
    out["Total_Income"] = out["Applicant_Income"] + out["Coapplicant_Income"]
    out["Loan_to_Income"] = out["Loan_Amount"] / (out["Total_Income"] + 1.0)
    out["Expense_to_Income"] = out["Monthly_Expenses"] / (out["Total_Income"] + 1.0)
    out["Collateral_Coverage"] = out["Collateral_Value"] / (out["Loan_Amount"] + 1.0)
    out["DTI_Ratio_Sq"] = out["DTI_Ratio"] ** 2
    out["Credit_Score_Sq"] = out["Credit_Score"] ** 2
    return out

X_train_fe = engineer_features(X_train)
X_test_fe = engineer_features(X_test)

ENG_NUM_COLS = NUM_COLS + [
    "Total_Income", "Loan_to_Income", "Expense_to_Income",
    "Collateral_Coverage", "DTI_Ratio_Sq", "Credit_Score_Sq"
]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), ENG_NUM_COLS),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(drop="first"))]), CAT_COLS),
])

dt_model = Pipeline([
    ("prep", preprocessor),
    ("clf", DecisionTreeClassifier(criterion="gini", max_depth=6, min_samples_leaf=20, random_state=42)),
])
lr_model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(C=1.0, penalty="l2", max_iter=2000, random_state=42)),
])

dt_model.fit(X_train_fe, y_train)
lr_model.fit(X_train_fe, y_train)

In [3]:
def evaluate_and_plot(idx, name, model):
    preds = model.predict(X_test_fe)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()

    print(f"{idx}. {name} Results -> Accuracy: {acc*100:.2f}% | Precision: {prec*100:.2f}% | Recall: {rec*100:.2f}% | F1-score: {f1*100:.2f}%")

    # Confusion Matrix Heatmap
    mat = np.array([[tp, fn], [fp, tn]])
    labels = np.array([[f"TP: {tp}", f"FN: {fn}"], [f"FP: {fp}", f"TN: {tn}"]])
    plt.figure(figsize=(7.5, 6))
    sns.heatmap(mat, annot=labels, fmt="", cmap="YlGnBu",
                xticklabels=["Positive", "Negative"],
                yticklabels=["Positive", "Negative"])
    plt.title(f"Confusion Matrix Heatmap — {name}")
    plt.xlabel("Predicted labels")
    plt.ylabel("True labels")
    plt.tight_layout()
    plt.show()

    # Accuracy, Precision, Recall, F1-score Bar Chart
    plt.figure(figsize=(8, 4.8))
    metric_names = ["Accuracy", "Precision", "Recall", "F1-score"]
    metric_vals = [acc * 100, prec * 100, rec * 100, f1 * 100]
    bars = plt.bar(metric_names, metric_vals, color=["#1d91c0", "#225ea8", "#081d58", "#41b6c4"], width=0.5)
    plt.ylim(0, 105)
    plt.ylabel("Score (%)")
    plt.title(f"{idx}. {name} Results (Accuracy, Precision, Recall, F1-score)")
    for b, v in zip(bars, metric_vals):
        plt.text(b.get_x() + b.get_width()/2, v + 2, f"{v:.2f}%", ha="center", fontweight="bold")
    plt.tight_layout()
    plt.show()

evaluate_and_plot(1, "Decision Tree", dt_model)
evaluate_and_plot(2, "Logistic Regression", lr_model)

1. Decision Tree Results -> Accuracy: 81.20% | Precision: 83.80% | Recall: 88.41% | F1-score: 86.04%
2. Logistic Regression Results -> Accuracy: 82.80% | Precision: 85.42% | Recall: 88.94% | F1-score: 87.14%
